In [ ]:
# ----------------------------
# CNN PINN simulator
# ----------------------------

# Import necessary libraries
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
from scipy.ndimage import distance_transform_edt
import os
import time
import pandas as pd

import os
import json
from pathlib import Path
import pyvista as pv


In [ ]:
# ----------------------------
# CNN PINN Block Definitions
# ----------------------------

class ConvBlock(nn.Module):
        """Two 3x3 convs, each followed by GroupNorm + SiLU."""
        def __init__(self, in_ch, out_ch, groups=8):
            super().__init__()
            groups = min(groups, out_ch)
            self.net = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.GroupNorm(groups, out_ch),
                nn.SiLU(),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.GroupNorm(groups, out_ch),
                nn.SiLU(),
            )
        
        def forward(self, x):
            return self.net(x)
    
class DownBlock(nn.Module):
    """Stride-2 conv downsample + ConvBlock.
    Learned (strided-conv) downsampling instead of max-pooling -- it preserves more boundary-layer detail,
    which matters for near-wall gradients that impact physics loss"""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv2d(in_ch, in_ch, 4, stride=2, padding=1)
        self.block = ConvBlock(in_ch, out_ch)
    
    def forward(self, x):
        return self.block(self.down(x))

class UpBlock(nn.Module):
    """Transpose-conv upsample, concat skip connection, ConvBlock."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)
        self.block = ConvBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            # guards against off-by-one size mismatches on odd input dims
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)
    
# ----------------------------
# Spectral Convolution Layer for the bottleneck to solve pressure across the whole domain
# ----------------------------

class SpectralConv2d(nn.Module):
    """2D Fourier layer. This is the spectral convolution layer used in the bottleneck of the CNN PINN architecture."""
    def __init__(self, in_ch, out_ch, modes_h, modes_w):
        super().__init__()
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.modes_h = modes_h
        self.modes_w = modes_w
        self.scale = (1.0 / (in_ch * out_ch))
        self.weights = nn.Parameter(self.scale * torch.rand(in_ch, out_ch, modes_h, modes_w, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft_rfft2(x, norm="ortho") # (B, C, H, W//2+1), complex

        out_ft = torch.zeros(B, self.out_ch, H, W // 2 + 1, dtype=torch.cfloat, device=x.device)
        mh = min(self.modes_h, H)
        mw = min(self.modes_w, W // 2 + 1)

        out_ft[:, :, :mh, :mw] = torch.einsum(
            "bixy,ioxy->boxy", x_ft[:, :, :mh, :mw], self.weights[:, :, :mh, :mw]
        )

        return torch.fft.irfft2(out_ft, s=(H, W), norm="ortho")
    
class FNOBlock(nn.Module):
    """Fourier Neural Operator block. This is the bottleneck of the CNN PINN architecture."""
    def __init__(self, channels, modes_h=16, modes_w=16):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_h, modes_w)
        self.pointwise = nn.Conv2d(channels, channels, 1)
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.act = nn.SiLU()

    def forward(self, x):
        out = self.spectral(x) + self.pointwise(x)
        return self.act(self.norm(out))
        

In [ ]:
class CNN_PINN(nn.Module):
    """
    U-Net encoder-deecoder with an FNO bottleneck.
    
    Input: (B, 4, H, W) -- occupancy, SDF, x-coord, y-coord
    Output: (B, 4, H, W) -- u, v, p, T (fluid region values only, masked in solid region)

    Sized for H=100, W = 200 (25x50 bottleneck, small enough for FFT-based spectral convs to be cheap)
    """
    def __init__(self, in_ch=4, base_ch=32, n_fno_blocks=4, fno_modes=16):
        super().__init__()

        self.stem = ConvBlock(in_ch, base_ch)               # H x   W
        self.down1 = DownBlock(base_ch, base_ch * 2)        # H/2 x W/2
        self.down2 = DownBlock(base_ch * 2, base_ch * 4)    # H/4 x W/4

        self.bottleneck = nn.Sequential(
            *[FNOBlock(base_ch * 4, fno_modes, fno_modes) for _ in range(n_fno_blocks)]
        )

        self.up2 = UpBlock(base_ch * 4, base_ch * 2, base_ch * 2)  # H/2 x W/2
        self.up1 = UpBlock(base_ch * 2, base_ch, base_ch)
        self.head = nn.Conv2d(base_ch, 4, kernel_size=1)  # u, v, p, T

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.bottleneck(s2)
        u2 = self.up2(b, s1)
        u1 = self.up1(u2, s0)
        out = self.head(u1)
        return out

In [ ]:
class SpatialDerivative(nn.Module):
    """
    Calculates spatial derivative operators via fixed conv2d kernels using central difference approximations.
    Replicate-padded to get a valid derivative at the boundary. This is used to compute the physics loss for the PINN.
    """
    def __init__(self, dx: float, dy: float):
        super().__init__()
        assert abs(dx - dy) < 1e-9, "assumes square pixels (dx == dy)"
        self.h = dx
        
        ddx = torch.tensor([[0, 0, 0], [-1, 0, 1], [0, 0, 0]], dtype=torch.float32) / (2 * dx)
        ddy = torch.tensor([[0, -1, 0], [0, 0, 0], [0, 1, 0]], dtype=torch.float32) / (2 * dy)
        lap = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32) / (dx * dy)

        self.register_buffer("ddx", ddx.view(1, 1, 3, 3))
        self.register_buffer("ddy", ddy.view(1, 1, 3, 3))
        self.register_buffer("lap", lap.view(1, 1, 3, 3))

    def _conv_deriv(self, f, kernel):
        f = F.pad(f, (1, 1, 1, 1), mode="replicate")
        return F.conv2d(f, kernel)

    def d_dx(self, f):
        return self._conv_deriv(f, self.ddx)
    
    def d_dy(self, f):
        return self._conv_deriv(f, self.ddy)
    
    def laplacian(self, f):
        return self._conv_deriv(f, self.lap)

In [ ]:
class PhysicsLoss(nn.Module):
    """
    Computes PDE residual + boundary-condition losses for a single fixed
    set of fluid properties / boundary conditions.

    forward() returns a dict of individual loss terms, which can be weighted and summed to form the total loss.
    """

    def __init__(self, fluid_properties: dict, boundary_conditions: dict, domain_size=(2.0, 1.0), grid_shape=(200, 100)):
        super().__init__()
        W, H = grid_shape
        dx = domain_size[0] / W
        dy = domain_size[1] / H
        self.deriv = SpatialDerivative(dx, dy)

        self.rho = fluid_properties["rho"]
        self.mu = fluid_properties["mu"]
        self.k = fluid_properties["k"]
        self.cp = fluid_properties["cp"]
        self.nu = self.mu / self.rho  # kinematic viscosity
        self.alpha = self.k / (self.rho * self.cp)  # thermal diffusivity

        self.U_in = boundary_conditions["U_in"]
        self.T_in = boundary_conditions["T_in"]
        self.T_wall = boundary_conditions["T_wall"]
        self.P_pin = boundary_conditions["P_pin"]

    def dilate(self, mask, iterations=1):
        for _ in range(iterations):
            mask = F.pad(mask, (1, 1, 1, 1), mode="replicate")
            mask = F.max_pool2d(mask, kernel_size=3, stride=1, padding=0)
        return mask

    def erode(self, mask, iterations=1):
        return 1.0 - self.dilate(1.0 - mask, iterations=iterations)
    
    def masked_mse(self, field: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        diff2 = (field - target) ** 2 * mask
        denom = mask.sum().clamp(min=1.0)
        return diff2.sum() / denom

    def forward(self, pred: torch.Tensor, occupancy: torch.Tensor) -> dict:
        """
        Args:
            pred: (B, 4, H, W) network output -- channels [u, v, p, T]
            occupancy: (B, 1, H, W) -- 1 = solid, 0 = fluid
        
        Returns:
            dict of scalar loss tensors: pde_continuity, pde_momentum_x, pde_momentum_y,
            pde_energy, bc_inlet, bc_wall, bc_outlet, bc_pressure_pin
        """
        u, v, p, T = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3], pred[:, 3:4]
        fluid = 1-occupancy  # 1 = fluid, 0 = solid
        solid = occupancy

        # Masks
        wall_band = fluid * self.dilate(solid, iterations = 1)
        interior = self.erode(fluid, iterations=1)

        inlet_mask = torch.zeros_like(fluid)
        inlet_mask[:, :, :, 0] = fluid[:, :, :, 0]  # left edge
        outlet_mask = torch.zeros_like(fluid)
        outlet_mask[:, :, :, -1] = fluid[:, :, :, -1]  # right edge
        
        # PDE residuals
        du_dx, du_dy = self.deriv.d_dx(u), self.deriv.d_dy(u)
        dv_dx, dv_dy = self.deriv.d_dx(v), self.deriv.d_dy(v)
        dp_dx, dp_dy = self.deriv.d_dx(p), self.deriv.d_dy(p)
        dT_dx, dT_dy = self.deriv.d_dx(T), self.deriv.d_dy(T)

        continuity = du_dx + dv_dy
        momentum_x = (u * du_dx + v * du_dy) + (1 / self.rho) * dp_dx - self.nu * (self.deriv.laplacian(u))
        momentum_y = (u * dv_dx + v * dv_dy) + (1 / self.rho) * dp_dy - self.nu * (self.deriv.laplacian(v))
        energy = (u * dT_dx + v * dT_dy) - self.alpha * (self.deriv.laplacian(T))

        losses = {
            "pde_continuity": self.masked_mse(continuity, torch.zeros_like(continuity), interior),
            "pde_momentum_x": self.masked_mse(momentum_x, torch.zeros_like(momentum_x), interior),
            "pde_momentum_y": self.masked_mse(momentum_y, torch.zeros_like(momentum_y), interior),
            "pde_energy": self.masked_mse(energy, torch.zeros_like(energy), interior)
            }
        
        # Boundary condition losses
        inlet_loss = self.masked_mse(u, 0.0, inlet_mask) + self.masked_mse(v, 0.0, inlet_mask) + self.masked_mse(T, self.T_in, inlet_mask)
        outlet_loss = self.masked_mse(du_dx, 0.0, outlet_mask) + self.masked_mse(dv_dx, 0.0, outlet_mask) + self.masked_mse(dT_dx, 0.0, outlet_mask)

        # add wall and pressure bc losses later
        losses["bc_inlet"] = inlet_loss
        losses["bc_outlet"] = outlet_loss

        return losses

In [ ]:
class PINN_Simulator(nn.Module):
    def __init__(self, fluid_properties: dict, boundary_conditions: dict, domain_size=(2.0, 1.0), grid_shape=(200, 100)):
        super().__init__()
        self.fluid_properties = fluid_properties
        self.boundary_conditions = boundary_conditions
        self.domain_size = domain_size
        self.grid_shape = grid_shape

        self.model = CNN_PINN(in_ch=4, base_ch=32, n_fno_blocks=4, fno_modes=16)
        self.physics_loss = PhysicsLoss(fluid_properties, boundary_conditions, domain_size, grid_shape)
        self.weights = {
            "pde_continuity": 1.0,
            "pde_momentum_x": 1.0,
            "pde_momentum_y": 1.0,
            "pde_energy": 1.0,
            "bc_inlet": 1.0,
            "bc_outlet": 1.0
        }

    def build_input_tensor(occupancy_grid: np.ndarray, domain_size=(2.0,1.0)) -> torch.Tensor:
        """
        Convert a binary occupancy grid (1 = solid, 0 = fluid) into the
        multi-channel input tensor for the network.
    
        Channels:
            0: occupancy            (0 = fluid, 1 = solid)
            1: signed distance fn   (positive in fluid, negative in solid),
                                    normalized by the domain diagonal
            2: normalized x coord   in [0, 1]
            3: normalized y coord   in [0, 1]
    
        Coordinate + SDF channels matter because plain convolutions are
        translation-equivariant, but inlet/outlet/wall locations are fixed in
        the domain -- the network needs to know *where* it is, and the SDF gives
        a smooth signal near walls instead of a hard 0/1 jump.
    
        Args:
            occupancy_grid: (H, W) array, 1 = solid, 0 = fluid
            domain_size: (Lx, Ly) physical domain size in meters
    
        Returns:
            Tensor of shape (4, H, W), float32
        """
        occ = occupancy_grid.astype(np.float32)
        H, W = occ.shape
    
        dx = domain_size[0] / W
        dy = domain_size[1] / H
        assert abs(dx - dy) < 1e-9, "build_input_tensor assumes square pixels (dx == dy)"
    
        dist_to_solid = distance_transform_edt(occ == 0)  # fluid px -> nearest solid px
        dist_to_fluid = distance_transform_edt(occ == 1)  # solid px -> nearest fluid px
        sdf_px = np.where(occ == 0, dist_to_solid, -dist_to_fluid)
        sdf_m = sdf_px * dx
        diag = np.sqrt(domain_size[0] ** 2 + domain_size[1] ** 2)
        sdf_norm = (sdf_m / diag).astype(np.float32)
    
        y_coords, x_coords = np.meshgrid(
            np.linspace(0.0, 1.0, H, dtype=np.float32),
            np.linspace(0.0, 1.0, W, dtype=np.float32),
            indexing="ij",
        )
    
        input_np = np.stack([occ, sdf_norm, x_coords, y_coords], axis=0)
        return torch.from_numpy(input_np)
    
    def combine_losses(self, loss_dict: dict, weights: dict) -> torch.Tensor:
        total = 0.0
        for name, value in loss_dict.items():
            total += weights.get(name, 1.0) * value
        return total
    
    def load_dataset(self, config_dir: str, results_dir: str, domain_size: tuple = (2.0, 1.0), grid_shape: tuple = (200, 100)) -> Dataset:
        """
        Load a dataset of occupancy grids and corresponding fluid flow solutions.
        The dataset is expected to be in the form of .npz files containing:
            - occupancy: (H, W) array, 1 = fluid, 0 = solid
            - u: (H, W) array, x-velocity
            - v: (H, W) array, y-velocity
            - p: (H, W) array, pressure
            - T: (H, W) array, temperature
            Args:
                config_dir: directory containing the configuration files for the dataset
                results_dir: directory containing the results of the simulations
                domain_size: physical domain size in meters
                grid_shape: shape of the grid (W, H)

            Returns:
                Dataset: the loaded dataset formatted as a pandas DataFrame with columns for occupancy, u, v, p, T, and any other relevant metadata.
            """
        
    
    def run(self, occupancy_grid: np.ndarray, domain_size: tuple = (2.0, 1.0)):
        """
        Run the PINN simulator on a given occupancy grid.
    
        Args:
            occupancy_grid: (H, W) array, 1 = fluid, 0 = solid
            domain_size: (Lx, Ly) physical domain size in meters
    
        Returns:
            Tensor of shape (4, H, W), float32 -- predicted u, v, p, T fields
        """
        input_tensor = self.build_input_tensor(occupancy_grid, domain_size)
        input_tensor = input_tensor.unsqueeze(0)  # add batch dimension
        with torch.no_grad():
            output = self.forward(input_tensor)
        return output.squeeze(0)  # remove batch dimension

    def train_model(self, 
                    config_dir: str,
                    data_dir: str,
                    checkpoint_dir: str,
                    pinn_result_dir: str,
                    fluid_properties: dict,
                    boundary_conditions: dict,
                    domain_size: tuple = (2.0, 1.0),
                    grid_shape: tuple = (200, 100),
                    batch_size: int = 4,
                    num_epochs: int = 100,
                    warmup_epochs: int = 10,
                    ramp_epochs: int = 10,
                    learning_rate: float = 1e-4,
                    device: str = "cpu",
                    num_workers: int = 4
                    ):
        """
        Train the PINN simulator on a dataset of occupancy grids and corresponding fluid flow solutions.
        """
        device = device
        os.makedirs(checkpoint_dir, exist_ok=True)
        os.makedirs(pinn_result_dir, exist_ok=True)

        # Load dataset
        

In [ ]:
# ========================================================================
# 1. Read config.json -> native occupancy grid + threshold + domain size
# ========================================================================

def read_config(config_path: str | Path) -> dict:
    """
    Read the configuration files for the dataset and return a dictionary of parameters.
    """
    config_path = Path(config_path)
    with open(config_path, "r") as f:
        config = json.load(f)
    
    return {
        "occupancy_grid": np.array(config["occupancy_grid"], dtype=np.float32),
        "threshold": float(config["threshold"]),
        "grid_nx": int(config["grid_nx"]),
        "grid_ny": int(config["grid_ny"]),
        "domain_length": float(config["domain_length"]),
        "domain_height": float(config["domain_height"]),
    }

# ========================================================================
# 2. Read MOOSE Exodus II results file -> coordinates + u, v, p, T fields
# ========================================================================

def _extract_valid_blocks(block) -> list:
    """
    Recursively collect all non-empty leaf datasets from a PyVista MultiBlock tree.
    A block is 'valid' if it has at least one point and one cell.
    """
    datasets = []
    if isinstance(block, pv.MultiBlock):
        for i in range(block.n_blocks):
            datasets.extend(_extract_valid_blocks(block[i]))
    else:
        if block is None:
            return []
        if block.n_points == 0 or block.n_cells == 0:
            return []
        datasets.append(block)
    return datasets

def read_moose_results(
        exodus_path: str | Path,
) -> dict[str, np.ndarray]:
    """
    Read a MOOSE Exodus II results file and map the fields onto the
    reference grid defined by GeometryConfig.

    Parameters
    ----------
    exodus_path : str | Path
        Path to the Exodus II file.

    Returns
    -------
    dict with keys: 'coords', 'u', 'v', 'p', 'T'
    """
    # Read the Exodus II file using PyVista
    exodus_path = str(exodus_path)
    reader = pv.get_reader(exodus_path)
    reader.set_active_time_value(reader.time_values[-1])  # last time step
    mb = reader.read()

    # Filter out empty blocks before mergine - avoids PyVista warnings
    valid_blocks = _extract_valid_blocks(mb)
    if not valid_blocks:
        raise RuntimeError(
            f"No non-empty blocks found in {exodus_path}. "
            "The simulation may have failed or written an empty results file."
        )
    
    # Combine the valid blocks into a single mesh and convert cell data to point data
    mesh = pv.MultiBlock(valid_blocks).combine()
    mesh = mesh.cell_data_to_point_data()

    # Extract the relevant fields from the mesh
    coords = np.array(mesh.points, dtype=np.float64)  # shape (N, 3)
    vel = np.array(mesh["vel_"], dtype=np.float64)  # shape (N, 3)
    u = vel[:, 0]  # x-velocity
    v = vel[:, 1]  # y-velocity
    p = np.array(mesh["p"], dtype=np.float64)  # shape (N,)
    T = np.array(mesh["T"], dtype=np.float64)  # shape (N,)
    moose_results = {
        "coords": coords,
        "u": u,
        "v": v,
        "p": p,
        "T": T,
    }
    return moose_results

# =========================================================================
# 3. Upscale native occupancy grid and validate size match to MOOSE results
# =========================================================================

def upscale_occupancy_grid(occupancy_grid: np.ndarray, scale: int) -> np.ndarray:
    """subdivides each cell into scale x scale cells, preserving the occupancy value (0 or 1)"""
    return np.kron(occupancy_grid, np.ones((scale, scale), dtype=occupancy_grid.dtype))

# ========================================================================
# 4. Compute SDF and Normalized Coordinates
# ========================================================================

def compute_sdf(occupancy_grid: np.ndarray, dx: float, dy: float, domain_height: float) -> np.ndarray:
    """
    Physical-distance signed distance function: positive in fluid, negative in solid, normalized by domain_height.
    """
    fluid = occupancy_grid == 1
    dist_to_solid = distance_transform_edt(fluid, sampling=(dy, dx))  # distance from fluid px to nearest solid px
    dist_to_fluid = distance_transform_edt(~fluid, sampling=(dy, dx))
    sdf = np.where(fluid, dist_to_solid, -dist_to_fluid)
    return (sdf / domain_height).astype(np.float32)

def compute_normalized_coords(nx: int, ny: int, dx: float, dy: float, domain_height: float):
    """Cell-center x, y coordinates, both normalized by domain_height (y in [0,1] and x in [0, L/H])"""
    x = (np.arange(nx) + 0.5) * dx / domain_height
    y = (np.arange(ny) + 0.5) * dy / domain_height
    xx, yy = np.meshgrid(x, y) # each (ny, nx)
    return xx.astype(np.float32), yy.astype(np.float32)

# ========================================================================
# 5. Interpolate MOOSE results onto cell centers via 4-corner avg
# ========================================================================

def build_node_lookup(node_points: np.ndarray, nx: int, ny: int, dx: float, dy: float, rtol: float = 1e-3) -> dict:
    """
    Maps each mesh node onto its (grid_i, grid_j) grid-line intersection index,
    snapping by rounding to the nearest multiple of dx, dy. Valid because the mesh
    is locked to the occupancy grid (blocky boundary) and the grid is uniform.

    Returns: dict {(grid_i, grid_j): node_index}, grid_i on [0, nx], grid_j on [0, ny].
    """
    x, y = node_points[:, 0], node_points[:, 1]
    gi = np.round(x / dx).astype(int)
    gj = np.round(y / dy).astype(int)

    # sanity check: snapped coordinates should closely match actual node coords
    snap_err_x = np.abs(gi * dx - x)
    snap_err_y = np.abs(gj * dy - y)
    tol = rtol * max(dx, dy)
    if np.any(snap_err_x > tol) or np.any(snap_err_y > tol):
        raise ValueError("Node coordinates do not align with grid lines within tolerance.")
    
    lookup = {}
    for idx, (i, j) in enumerate(zip(gi, gj)):
        lookup[(i, j)] = idx
    return lookup

def interpolate_nodes_to_cell_centers(node_fields: dict, node_lookup: dict, nx: int, ny: int) -> dict:
    """
    for each cell (ix, iy), averages the 4 corner nodes 
    (ix, iy), (ix+1, iy), (ix, iy+1), (ix+1, iy+1) to get the cell-center value.
    """
    field_names = [k for k in node_fields.keys() if k != "coords"]
    out = {name: np.full((ny, nx), np.nan, dtype=np.float32) for name in field_names}

    for iy in range(ny):
        for ix in range(nx):
            corners = [(ix, iy), (ix + 1, iy), (ix, iy + 1), (ix + 1, iy + 1)]
            node_ids = [node_lookup.get(corner) for corner in corners]
            if any(n is None for n in node_ids):
                continue  # skip cells with missing corners
            for name in field_names:
                vals = node_fields[name][node_ids]
                out[name][iy, ix] = np.mean(vals)
    return out

# ========================================================================
# 6. Assemble the tidy per-cell dataframe from the MOOSE results and the occupancy grid
# ========================================================================

def assemble_dataframe(geometry_id: str, occupancy: np.ndarray, sdf: np.ndarray, x: np.ndarray, y: np.ndarray,
                       cell_fields: dict) -> pd.DataFrame:
    """
    Flattens (ny, nx) arrays into a tidy dataframe: one row per cell.
    Columns: geometry_id, ix, iy, occupancy, sdf, x, y, u, v, p, T.
    """
    ny, nx = occupancy.shape
    ix_grid, iy_grid = np.meshgrid(np.arange(nx), np.arange(ny))

    data = {
        "geometry_id": geometry_id,
        "ix": ix_grid.flatten(),
        "iy": iy_grid.flatten(),
        "occupancy": occupancy.flatten(),
        "sdf": sdf.flatten(),
        "x": x.flatten(),
        "y": y.flatten(),
    }
    for name, field in cell_fields.items():
        data[name] = field.flatten()
    return pd.DataFrame(data)

# ========================================================================
# 7. Top-level: build_dataset
# ========================================================================

def build_dataset(config_dir: str, results_dir: str, sim_scale: int, output_dir: str, overwrite: bool=False) -> dict:
    """
    For each geometry in config_dir, read the occupancy grid and MOOSE results, upscale the occupancy grid,
    compute the SDF and normalized coordinates, interpolate the MOOSE results onto cell centers, 
    and assemble the data into a tidy dataframe. Save the dataframe as a CSV file in output_dir.
    """ 
    config_dir = Path(config_dir)
    results_dir = Path(results_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    summary = {"built": [], "skipped": [], "failed": []}

    file_root = "hx_200x100-0.010m-t"

    num_geometries = len(list(config_dir.glob("*.json")))
    for i in range(num_geometries):
        geometry_id = f"{file_root}{i:04d}"
        config_path = config_dir / f"{geometry_id}.json"
        results_path = results_dir / f"{geometry_id}.e"
        output_path = output_dir / f"{geometry_id}.parquet"

        if output_path.exists() and not overwrite:
            summary["skipped"].append(geometry_id)
            continue

        try:
            # Read config.json -> native occupancy grid + threshold + domain size
            config = read_config(config_path)
            occupancy_grid = config["occupancy_grid"]
            Lx, Ly = config["domain_length"], config["domain_height"]
            grid_nx, grid_ny = config["grid_nx"], config["grid_ny"]
            dx, dy = Lx / grid_nx, Ly / grid_ny

            # Upscale occupancy grid if sim_scale > 1 and compute binary occupancy
            if sim_scale > 1:
                occupancy_grid = upscale_occupancy_grid(occupancy_grid, sim_scale)
            occupancy_binary = (occupancy_grid > config["threshold"]).astype(np.float32)

            # Compute SDF and normalized coordinates
            ny_target, nx_target = occupancy_binary.shape
            sdf = compute_sdf(occupancy_binary, dx, dy, Ly)
            x_coords, y_coords = compute_normalized_coords(nx_target, ny_target, dx, dy, Ly)

            # Read MOOSE results and interpolate onto cell centers
            moose_results = read_moose_results(results_path)
            node_lookup = build_node_lookup(moose_results["coords"], nx_target, ny_target, dx, dy)
            cell_fields = interpolate_nodes_to_cell_centers(moose_results, node_lookup, nx_target, ny_target)

            df = assemble_dataframe(geometry_id, occupancy_binary, sdf, x_coords, y_coords, cell_fields)
            df.to_parquet(output_path)

            summary["built"].append(geometry_id)
        except Exception as e:
            summary["failed"].append((geometry_id, f"Config read error: {e}"))
            print(f"Failed to process {geometry_id}: {e}")

    return summary


In [ ]:
summary = build_dataset(
    config_dir="configs",
    results_dir="results",
    sim_scale=1,
    output_dir="dataset",
    overwrite=False
)

print("Dataset build summary:")
for category, geometries in summary.items():
    print(f"  {category.capitalize()}: {len(geometries)}")
